In [1]:
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
data_path = "./data/qa_slim.jsonl"
import os
os.path.exists(data_path)

True

In [3]:
import json

# Load the dataset
with open(data_path, 'r') as f:
    dataset = [json.loads(line) for line in f]

print(f"Dataset size: {len(dataset)} examples")
print(f"\nDataset columns: {dataset[0].keys()}")
print("\n" + "="*60)
print("Sample entry:")
print("="*60)

# Show a sample
sample = dataset[0]
print(f"\nQuestion: {sample['question']}")
print(f"\nAnswer: {sample['answer']}")

Dataset size: 2190 examples

Dataset columns: dict_keys(['question', 'answer'])

Sample entry:

Question: What is the primary function of the "intra-oral camera" mentioned by David Plotz?

Answer: It is a tool used by dentists to magnify dental flaws on a screen.


In [4]:
import json


def convert_to_messages(example):
    return {
        "messages": [
            {"role": "user", "content": example['question']},
            {"role": "assistant", "content": example['answer']}
        ]
    }

# Show an example of the converted format
sample_converted = convert_to_messages(dataset[0])
print("Converted format:")
print(json.dumps(sample_converted, indent=2))

Converted format:
{
  "messages": [
    {
      "role": "user",
      "content": "What is the primary function of the \"intra-oral camera\" mentioned by David Plotz?"
    },
    {
      "role": "assistant",
      "content": "It is a tool used by dentists to magnify dental flaws on a screen."
    }
  ]
}


In [13]:
VAL_SIZE = 100
TEST_SIZE = 100  # will be used for evaluation
TRAIN_SIZE = len(dataset) - VAL_SIZE - TEST_SIZE

# Shuffle and select a subset
import random
SEED = 42
random.seed(SEED)
random.shuffle(dataset)
train_dataset = dataset[:TRAIN_SIZE]
val_dataset = dataset[TRAIN_SIZE:TRAIN_SIZE+VAL_SIZE]
test_dataset = dataset[TRAIN_SIZE+VAL_SIZE:]


In [15]:

# Convert to messages format
train_data = [convert_to_messages(example) for example in train_dataset]
val_data = [convert_to_messages(example) for example in val_dataset]
test_data = [convert_to_messages(example) for example in test_dataset]

print(f"Training examples: {len(train_data)}")
print(f"Validation examples: {len(val_data)}")
print(f"Test examples: {len(test_data)}")

Training examples: 1990
Validation examples: 100
Test examples: 100


In [16]:
train_data[0]

{'messages': [{'role': 'user',
   'content': "What does the persistence of Falwell's belief highlight about the situation described in the document?"},
  {'role': 'assistant',
   'content': 'It highlights the tension between public apology and private conviction.'}]}

In [19]:
TULU_MIX_SIZE = int(3/7 * TRAIN_SIZE)  # 30% tulu 70% new-knowledge
from datasets import load_dataset

tulu_dataset = load_dataset("allenai/tulu-3-sft-mixture", split="train", streaming=True)
tulu_dataset = tulu_dataset.shuffle(seed=SEED)
for i, x in enumerate(tulu_dataset):
    if i >= TULU_MIX_SIZE: break
    train_data.append({'messages': x['messages']})

train_data[-1]


{'messages': [{'content': 'If the product of $\\left(x+k\\right)\\left(x-4\\right)$ does not contain a linear term of $x$, then the value of $k$ is ____.',
   'role': 'user'},
  {'content': 'To solve this problem, we need to expand the product \\((x + k)(x - 4)\\) and identify the condition under which the linear term of \\(x\\) vanishes.\n\n### Steps:\n\n1. **Expand the product**: Expand the expression \\((x + k)(x - 4)\\) to identify the linear term in \\(x\\).\n2. **Eliminate the linear term**: Set the coefficient of the linear term to zero and solve for \\(k\\).\n\nLet\'s implement this step-by-step in Python using SymPy:\n\n```python\nimport sympy as sp\n\n# Define the variables\nx, k = sp.symbols(\'x k\')\n\n# Define the expression (x + k)(x - 4)\nexpression = (x + k)*(x - 4)\n\n# Expand the expression\nexpanded_expression = sp.expand(expression)\n\n# Display the expanded expression\nprint(f"Expanded Expression: {expanded_expression}")\n\n# Collect the coefficients of x\ncoeffici

In [21]:
print(f'Training data size: {len(train_data)}')

Training data size: 2842


In [22]:
# output dirs
from pathlib import Path
OUTPUT_DIR = Path('./knowledge-ingestion-test')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# save to jsonl
train_file = OUTPUT_DIR / 'train.jsonl'
val_file = OUTPUT_DIR / 'val.jsonl'
test_file = OUTPUT_DIR / 'test.jsonl'

with open(train_file, 'w') as f:
    for item in train_data:
        f.write(json.dumps(item) + '\n')

with open(val_file, 'w') as f:
    for item in val_data:
        f.write(json.dumps(item) + '\n')

with open(test_file, 'w') as f:
    for item in test_data:
        f.write(json.dumps(item) + '\n')


print(f"Training data saved to {train_file}")
print(f"Validation data saved to {val_file}")
print(f"Test data saved to {test_file}")

Training data saved to knowledge-ingestion-test/train.jsonl
Validation data saved to knowledge-ingestion-test/val.jsonl
Test data saved to knowledge-ingestion-test/test.jsonl


In [23]:
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

train_kwargs = {
    "model_path": MODEL_NAME,
    "data_path": str(train_file),
    "ckpt_output_dir": OUTPUT_DIR / 'model',
    "dataset_type": "chat_template",
    "field_messages": "messages",
    "num_epochs": 1,
    "learning_rate": 1.5e-4,
    "micro_batch_size": 16,
    "max_seq_len": 1024,
    "seed": 42,
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.1,
    "load_in_4bit": False,
    "bf16": True,
    "sample_packing": False,
    "logging_steps": 10,
    "eval_steps": 10,
    "save_steps": 10,
    "save_total_limit": 1,
    "wandb_project": "sunday-auto-customize",
    "wandb_entity": "ronny21",
    "wandb_run_name": "knowledge-ingestion-test",
}

In [24]:
from training_hub.algorithms.lora import UnslothLoRABackend, JSONLLoggingCallback

backend = UnslothLoRABackend()

In [25]:
model, tokenizer = backend._load_unsloth_model(train_kwargs)

/workspace/home/lab/rawhad/auto-customize/.venv/lib/python3.12/site-packages/training_hub/algorithms/lora.py:167: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 8. Max memory: 79.097 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

unsloth/Qwen3-4B-Instruct-2507 does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [26]:
model = backend._apply_lora_config(model, train_kwargs)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.4 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [27]:
train_dataset = backend._prepare_dataset(train_kwargs, tokenizer)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2842 [00:00<?, ? examples/s]

In [28]:
validation_kwargs = dict(train_kwargs)
validation_kwargs["data_path"] = str(val_file)
validation_dataset = backend._prepare_dataset(validation_kwargs, tokenizer)


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [29]:
train_kwargs['data_path'], validation_kwargs['data_path']

('knowledge-ingestion-test/train.jsonl', 'knowledge-ingestion-test/val.jsonl')

In [30]:
training_args = backend._build_training_args(train_kwargs)
training_args.do_eval = True
training_args.eval_strategy = "steps"
training_args

UnslothSFTConfig(output_dir='knowledge-ingestion-test/model', per_device_train_batch_size=16, num_train_epochs=1, max_steps=-1, learning_rate=0.00015, lr_scheduler_type=<SchedulerType.LINEAR: 'linear'>, lr_scheduler_kwargs=None, warmup_steps=10, optim=<OptimizerNames.ADAMW_8BIT: 'adamw_8bit'>, optim_args=None, weight_decay=0.01, adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-08, optim_target_modules=None, gradient_accumulation_steps=1, average_tokens_across_devices=True, max_grad_norm=0.3, label_smoothing_factor=0.0, bf16=True, fp16=False, bf16_full_eval=False, fp16_full_eval=False, tf32=None, gradient_checkpointing=True, gradient_checkpointing_kwargs=None, torch_compile=False, torch_compile_backend=None, torch_compile_mode=None, use_liger_kernel=False, liger_kernel_config=None, use_cache=False, neftune_noise_alpha=None, torch_empty_cache_steps=250, auto_find_batch_size=False, logging_strategy=<IntervalStrategy.STEPS: 'steps'>, logging_steps=10, logging_first_step=False, log_on_each

In [31]:
training_args.per_device_eval_batch_size = 1

In [32]:
from trl import SFTTrainer

In [33]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    callbacks=[JSONLLoggingCallback(train_kwargs["ckpt_output_dir"])],
)
trainer

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/2842 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/100 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [34]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,842 | Num Epochs = 1 | Total steps = 178
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 1 x 1) = 16
 "-____-"     Trainable parameters = 16,515,072 of 4,038,983,168 (0.41% trained)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: ronny21 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss,Validation Loss
10,2.004638,4.441634
20,1.135295,2.875918
30,0.814855,2.514306
40,0.946036,2.372106
50,0.686638,2.298966
60,0.868821,2.224810
70,0.785394,2.162085
80,0.711818,2.122126
90,0.692987,2.083025
100,0.657282,2.051134


eval/loss,█▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁
eval/runtime,▆▁▂▂▃▂▂█▄▄▆▃▁▁▂▂▂
eval/samples_per_second,▃█▇▇▆▇▇▁▅▄▃▆██▇▇▆
eval/steps_per_second,▃█▇▇▆▇▇▁▅▄▃▆██▇▇▆
train/epoch,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇███
train/global_step,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇███
train/grad_norm,█▃▁▃▁▁▁▂▂▂▁▁▂▂▂▃▂
train/learning_rate,███▇▇▆▆▅▅▄▄▃▃▂▂▁▁
train/loss,█▄▂▃▁▂▂▂▁▁▂▂▂▂▂▂▁
eval/loss,1.95451
eval/runtime,6.2855


TrainOutput(global_step=178, training_loss=0.8440368496969844, metrics={'train_runtime': 201.5537, 'train_samples_per_second': 14.1, 'train_steps_per_second': 0.883, 'total_flos': 1.4371784242541568e+16, 'train_loss': 0.8440368496969844, 'epoch': 1.0})